# Phase 1 — Generate Augmented Training Data

**Pipeline**: FAISS few-shot retrieval → Qwen3-8B (4-bit) → character-span mapping

**Colab free tier**: T4 (15 GB VRAM). Model loads in ~5 GB via 4-bit quantisation.
Checkpoint is saved to Google Drive so progress survives disconnects.

**Estimated throughput**: ~1–2 s/pair on T4 → set `SAMPLE_SIZE=1_000` for ~4 h run.
Increase to 5 000 on Colab Pro (A100, ~30 min).

**Steps**:
1. Run **Cell 1** (install) → restart runtime when prompted
2. Mount Google Drive (Cell 2b)
3. Run **Cell 2** (HF login) → paste token
4. Verify data files (Cell 3)
5. Run all remaining cells top-to-bottom
6. Call `main()` (last cell) — safe to interrupt and resume


In [1]:
import subprocess, sys

pkgs = [
    "numpy",
    "pyarrow",
    "transformers>=5.5.0",
    "accelerate",           # required for device_map=auto
    "bitsandbytes",         # required for 4-bit quantisation on GPU
    "sentence-transformers",
    "faiss-cpu",
    "rapidfuzz",
    "tqdm",
    "pandas",
]

print("Installing packages (~2 min first run) ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", *pkgs])
print("✓ Installation complete")
print()
print("━" * 60)
print("  ⚠  Restart runtime now:  Runtime → Restart session")
print("  Then re-run all cells from the top.")
print("━" * 60)


Installing packages (~2 min first run) ...


✓ Installation complete

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ⚠  Restart runtime now:  Runtime → Restart session
  Then re-run all cells from the top.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [2]:
# Hugging Face login — required to download the model
# Get token at: https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()


In [3]:
# Files save locally to /content/ppt_phase1/
# Auto-downloaded to your machine when main() completes.
import os
DRIVE_DIR = "/content/ppt_phase1"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✓ Output dir: {DRIVE_DIR}")
print("  Files will auto-download when main() finishes.")


In [4]:
# Verify competition data files are present
# Upload via: Colab file browser, or kaggle API (see README)
import os
for f in ['features.csv', 'patient_notes.csv', 'train.csv']:
    status = "✓ found" if os.path.exists(f) else "✗ MISSING — upload before running main()"
    print(f"  {f}: {status}")

  features.csv: ✓ found
  patient_notes.csv: ✓ found
  train.csv: ✓ found


## Configuration

Edit the keys below before running. Most defaults are fine for Colab T4.

| Key | Default | When to change |
|-----|---------|----------------|
| `SAMPLE_SIZE` | `10_000` | Lower to `1_000` for a quick smoke test |
| `GPU_MEM_UTIL` | `0.82` | Lower to `0.75` if you hit OOM during vLLM init |
| `MAX_MODEL_LEN` | `4096` | Lower to `2048` if OOM persists |
| `FUZZY_SCORE_CUTOFF` | `72` | Raise to reduce false-positive span matches |

In [5]:
import ast, gc, json, logging, os, re, sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from rapidfuzz import fuzz, process as rfprocess
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm   # auto picks notebook-friendly bar

# ── Paths ─────────────────────────────────────────────────────────────────────
# DRIVE_DIR set in the Drive-mount cell above. Fallback for non-Colab.
try:
    _DRIVE = Path(DRIVE_DIR)
except NameError:
    _DRIVE = Path(".")   # running locally

CONFIG = {
    # Input data (upload to Colab or symlink from Drive)
    "DATA_DIR":              Path("."),
    # Output + checkpoint on Drive so they survive disconnect
    "OUTPUT_FILE":           _DRIVE / "augmented_train.csv",
    "CHECKPOINT_FILE":       _DRIVE / "augmented_train_checkpoint.csv",
    "CHECKPOINT_EVERY":      50,    # save every N rows (smaller = safer vs Colab cuts)
    # FAISS cache on Drive too
    "FAISS_INDEX_FILE":      _DRIVE / "faiss_features.index",
    "FAISS_META_FILE":       _DRIVE / "faiss_metadata.parquet",
    # ── Sampling ──────────────────────────────────────────────────────────────
    # 1 000 notes × ~20 features ≈ 20 000 pairs ≈ 6–8 h on T4 (free)
    # 500 notes ≈ 3–4 h; 5 000 notes on Colab Pro A100 ≈ 1 h
    "SAMPLE_SIZE":           500,
    "RANDOM_SEED":           42,
    # ── Embedding ─────────────────────────────────────────────────────────────
    "EMBED_MODEL":           "all-MiniLM-L6-v2",
    "EMBED_BATCH_SIZE":      64,
    "TOP_K_EXAMPLES":        3,
    # ── LLM — 4-bit Qwen3-8B fits T4 (15 GB) in ~5 GB, no special libs needed ──
    "LLM_MODEL":             "Qwen/Qwen3-8B",
    "DRAFT_MODEL":           "Qwen/Qwen3-0.6B",  # speculative decoding draft — same family, same tokenizer,
    "LLM_4BIT":              True,   # set False if on A100 / H100 with plenty VRAM
    "MAX_NEW_TOKENS":        256,
    # ── Span matching ─────────────────────────────────────────────────────────
    "FUZZY_SCORE_CUTOFF":    72,
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print("✓ Imports and CONFIG loaded")
print(f"  Output  → {CONFIG['OUTPUT_FILE']}")
print(f"  Checkpoint → {CONFIG['CHECKPOINT_FILE']}")


✓ Imports and CONFIG loaded
  Output  → augmented_train.csv
  Checkpoint → augmented_train_checkpoint.csv


## Section 1 — Data Loading & Filtering

Loads `train.csv`, `features.csv`, and `patient_notes.csv`.

Filters out the ~14,300 notes that are already annotated in `train.csv` — we only
want to pseudo-label the **unannotated** portion of the 42,146-note corpus.
Then randomly samples `SAMPLE_SIZE` (default 10,000) of those unannotated notes.

In [6]:
def load_and_filter_data(cfg: dict) -> tuple:
    """
    Returns
    -------
    sample_notes : pd.DataFrame  — unannotated notes sampled for labelling
    train_df     : pd.DataFrame  — train.csv with annotation/location as Python lists
    features_df  : pd.DataFrame  — features.csv
    pn_df        : pd.DataFrame  — full patient_notes.csv (needed for FAISS metadata)
    """
    log.info("Loading CSV files ...")
    data_dir    = cfg["DATA_DIR"]
    train_df    = pd.read_csv(data_dir / "train.csv")
    features_df = pd.read_csv(data_dir / "features.csv")
    pn_df       = pd.read_csv(data_dir / "patient_notes.csv")

    def safe_parse_list(val):
        if pd.isna(val):
            return []
        try:
            result = ast.literal_eval(str(val))
            return result if isinstance(result, list) else []
        except (ValueError, SyntaxError):
            return []

    train_df["annotation"] = train_df["annotation"].apply(safe_parse_list)
    train_df["location"]   = train_df["location"].apply(safe_parse_list)

    annotated_pn_nums = set(train_df["pn_num"].unique())
    log.info(f"  Annotated notes in train.csv : {len(annotated_pn_nums)}")

    unannotated = pn_df[
        ~pn_df["pn_num"].isin(annotated_pn_nums)
        & pn_df["pn_history"].notna()
        & (pn_df["pn_history"].str.strip() != "")
    ].copy()
    log.info(f"  Unannotated notes available  : {len(unannotated)}")

    n_sample     = min(cfg["SAMPLE_SIZE"], len(unannotated))
    sample_notes = unannotated.sample(n=n_sample, random_state=cfg["RANDOM_SEED"]).reset_index(drop=True)
    log.info(f"  Sampled for labelling        : {len(sample_notes)}")

    return sample_notes, train_df, features_df, pn_df

print("✓ Section 1: load_and_filter_data defined")

✓ Section 1: load_and_filter_data defined


## Section 2 — FAISS Vector Index (Few-Shot Retrieval)

Builds a FAISS `IndexFlatIP` (inner-product / cosine similarity) over all annotated
examples in `train.csv`. Each vector encodes `"Feature: <text>  Annotation: <text>"`.

At generation time, for each `(feature, note)` pair we retrieve the **3 most similar**
labelled examples to use as few-shot context in the LLM prompt.

> **Caching**: The index is saved to `faiss_features.index` + `faiss_metadata.parquet`
> on first build. Re-running the notebook skips the rebuild and loads from disk in seconds.

In [7]:
def build_faiss_index(train_df, pn_df, features_df, cfg) -> tuple:
    idx_path  = cfg["FAISS_INDEX_FILE"]
    meta_path = cfg["FAISS_META_FILE"]

    if idx_path.exists() and meta_path.exists():
        log.info("Loading cached FAISS index ...")
        index    = faiss.read_index(str(idx_path))
        metadata = pd.read_parquet(meta_path).to_dict("records")
        log.info(f"  Loaded {index.ntotal} vectors (dim={index.d})")
        return index, metadata

    log.info("Building FAISS index from train.csv ...")
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = features_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()

    embed_texts, metadata = [], []
    for _, row in train_df.iterrows():
        feature_text   = feat_map.get((row["case_num"], row["feature_num"]), "")
        pn_history     = pn_map.get(row["pn_num"], "")
        annotation_str = " | ".join(a for a in row["annotation"] if isinstance(a, str) and a.strip())
        if not feature_text or not annotation_str:
            continue
        embed_texts.append(f"Feature: {feature_text}  Annotation: {annotation_str}")
        metadata.append({
            "feature_text": feature_text,
            "annotation":   annotation_str,
            "pn_history":   (pn_history or "")[:500],
            "location":     str(row["location"]),
        })

    log.info(f"  Embedding {len(embed_texts)} train examples ...")
    embed_model = SentenceTransformer(cfg["EMBED_MODEL"])
    embeddings  = embed_model.encode(
        embed_texts, batch_size=cfg["EMBED_BATCH_SIZE"],
        show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    del embed_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    log.info(f"  FAISS index: {index.ntotal} vectors, dim={dim}")
    faiss.write_index(index, str(idx_path))
    pd.DataFrame(metadata).to_parquet(meta_path, index=False)
    log.info("  Index cached to disk.")
    return index, metadata


def retrieve_few_shot_examples(query_feature_text, index, metadata, embed_model, top_k=3) -> list:
    query_vec = embed_model.encode(
        [f"Feature: {query_feature_text}  Annotation:"],
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    distances, indices = index.search(query_vec, top_k)
    return [metadata[i] for i in indices[0] if 0 <= i < len(metadata)]

print("✓ Section 2: build_faiss_index, retrieve_few_shot_examples defined")

✓ Section 2: build_faiss_index, retrieve_few_shot_examples defined


## Section 3 — Prompt Construction

Each prompt is a 3-message chat conversation:
1. **System**: instructs the model to extract verbatim spans as JSON
2. **User**: includes 3 FAISS-retrieved examples + the target note + the target feature
3. *(Assistant turn added at generation time by vLLM)*

The `/no_think` suffix disables Qwen3.5's chain-of-thought mode so the model
outputs clean JSON without wrapping `<think>...</think>` blocks.

In [8]:
SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation, no <think> blocks.\n"
    'Output format: {"spans": ["exact text 1", "exact text 2"]}'
)


def build_messages(feature_text, pn_history, few_shot_examples) -> list:
    examples_block = ""
    for i, ex in enumerate(few_shot_examples, start=1):
        note_excerpt = ex["pn_history"][:300].replace("\n", " ").strip()
        ann_parts    = [a.strip() for a in ex["annotation"].split(" | ") if a.strip()]
        ann_json     = json.dumps(ann_parts)  # valid JSON with double-quoted strings
        examples_block += (
            f"\n[Example {i}]\n"
            f"Note (excerpt): \"{note_excerpt}\"\n"
            f"Feature: {ex['feature_text']}\n"
            f'Answer: {{"spans": {ann_json}}}\n'
        )

    target_note  = pn_history.replace("\n", " ").strip()
    user_content = (
        f"Here are labelled examples:{examples_block}\n"
        f"---\n"
        f"Now label this note.\n"
        f"Note: \"{target_note}\"\n"
        f"Feature: {feature_text}\n\n"
        "/no_think"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

print("✓ Section 3: SYSTEM_PROMPT, build_messages defined")

✓ Section 3: SYSTEM_PROMPT, build_messages defined


## Section 4 — Span → Character Position Mapping

The LLM outputs text strings like `"substernal pressure"`. The competition requires
character offsets like `"42 62"`. This section maps extracted strings back to their
exact position in the original patient note.

**Three-step strategy** (most accurate first):
1. Exact substring match
2. Case-insensitive exact match
3. `rapidfuzz` sliding window — handles minor spacing/casing differences

In [9]:
def _strip_think_tokens(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def find_span_locations(span_texts, pn_history, fuzzy_cutoff=72) -> list:
    if not span_texts or not pn_history:
        return []

    locations, pn_lower = [], pn_history.lower()

    for span in span_texts:
        span = span.strip()
        if not span:
            continue

        # 1. Exact match
        idx = pn_history.find(span)
        if idx != -1:
            locations.append(f"{idx} {idx + len(span)}")
            continue

        # 2. Case-insensitive
        idx = pn_lower.find(span.lower())
        if idx != -1:
            locations.append(f"{idx} {idx + len(span)}")
            continue

        # 3. Fuzzy sliding window
        span_len = len(span)
        min_win  = max(1, int(span_len * 0.80))
        max_win  = min(len(pn_history), int(span_len * 1.20))

        best_score, best_start, best_end = 0, -1, -1
        for win_size in range(min_win, max_win + 1):
            n_windows  = len(pn_history) - win_size + 1
            if n_windows <= 0:
                continue
            candidates = [pn_history[s: s + win_size] for s in range(n_windows)]
            result = rfprocess.extractOne(span, candidates, scorer=fuzz.ratio, score_cutoff=fuzzy_cutoff)
            if result is not None:
                _text, score, pos = result
                if score > best_score:
                    best_score, best_start, best_end = score, pos, pos + win_size

        if best_score >= fuzzy_cutoff and best_start != -1:
            locations.append(f"{best_start} {best_end}")

    return locations

print("✓ Section 4: _strip_think_tokens, find_span_locations defined")

✓ Section 4: _strip_think_tokens, find_span_locations defined


## Section 5 — LLM Initialisation

Loads `Qwen/Qwen2.5-7B-Instruct` in **4-bit NF4** quantisation via `bitsandbytes`.

- VRAM footprint: ~5 GB (fits T4 free tier)
- `device_map="auto"`: spreads layers across GPU + CPU if needed
- Set `LLM_4BIT: False` in CONFIG to use float16 on A100/H100


In [10]:
def init_llm(cfg: dict):
    """Load target + draft model for speculative decoding (~5.5 GB total on T4)."""
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    log.info(f"Loading tokenizer: {cfg['LLM_MODEL']} ...")
    tokenizer = AutoTokenizer.from_pretrained(cfg["LLM_MODEL"], trust_remote_code=True)

    def _load_model(model_id, label):
        if cfg.get("LLM_4BIT", True) and torch.cuda.is_available():
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
            m = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_cfg,
                device_map="auto",
                trust_remote_code=True,
            )
        else:
            m = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True,
            )
        m.eval()
        mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        log.info(f"  {label} loaded | GPU mem so far: {mem:.1f} GB")
        return m

    log.info("Loading target model ...")
    model = _load_model(cfg["LLM_MODEL"], "Target")

    draft_model = None
    if cfg.get("DRAFT_MODEL"):
        log.info(f"Loading draft model: {cfg['DRAFT_MODEL']} ...")
        try:
            draft_model = _load_model(cfg["DRAFT_MODEL"], "Draft")
            # store input device same way as target
            try:
                draft_model._input_device = next(draft_model.parameters()).device
            except StopIteration:
                draft_model._input_device = torch.device("cpu")
            log.info("✓ Speculative decoding enabled")
        except Exception as e:
            log.warning(f"Draft model failed ({e}) — falling back to standard decoding")
            draft_model = None

    try:
        model._input_device = next(model.parameters()).device
    except StopIteration:
        model._input_device = torch.device("cpu")

    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    log.info(f"✓ Models ready | total GPU mem: {mem:.1f} GB")
    return model, tokenizer, draft_model


def make_sampling_params(cfg: dict) -> dict:
    return {
        "max_new_tokens": cfg["MAX_NEW_TOKENS"],
        "do_sample":      False,
    }

print("✓ Section 5: init_llm, make_sampling_params defined")


✓ Section 5: init_llm, make_sampling_params defined


## Section 6 — Main Generation Loop

Three phases:

**Phase A — Build prompts**: For each `(unannotated note × feature)` pair, retrieve 3 FAISS
examples and build the chat message list. This produces all `N` prompts up front.

**Phase B — Batch LLM inference**: Send all prompts through `llm.chat()` in batches of 256.
vLLM's continuous batching means larger batches = better GPU utilisation.

**Phase C — Parse + map spans**: Parse the JSON output from each prediction, then map the
extracted text strings to `"start end"` character offsets using Section 4's logic.

In [11]:
def _load_checkpoint(cfg: dict) -> tuple:
    """Return (done_ids set, existing_rows list). Empty if no checkpoint."""
    ckpt = Path(cfg["CHECKPOINT_FILE"])
    if ckpt.exists():
        df = pd.read_csv(ckpt)
        log.info(f"  Checkpoint found: {len(df)} rows already done — resuming.")
        return set(df["id"].tolist()), df.to_dict("records")
    return set(), []


def _save_checkpoint(rows: list, cfg: dict):
    """Write atomically: write to .tmp then rename so crash can't corrupt."""
    dst = Path(cfg["CHECKPOINT_FILE"])
    tmp = dst.with_suffix(".tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    tmp.replace(dst)   # atomic on POSIX; on Windows: replaces existing


def _infer_one(msgs, model, tokenizer, sampling_params, draft_model=None) -> str:
    """Run one inference with optional speculative decoding. Safe GPU cleanup."""
    device = getattr(model, "_input_device",
                     torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    text    = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs  = tokenizer(text, return_tensors="pt").to(device)
    out_ids = None
    try:
        with torch.no_grad():
            gen_kwargs = dict(
                **inputs,
                max_new_tokens=sampling_params["max_new_tokens"],
                do_sample=sampling_params["do_sample"],
                pad_token_id=tokenizer.eos_token_id,
            )
            if draft_model is not None:
                gen_kwargs["assistant_model"] = draft_model   # speculative decoding
            out_ids = model.generate(**gen_kwargs)
        return tokenizer.decode(
            out_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )
    finally:
        del inputs
        if out_ids is not None:
            del out_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def _parse_spans(raw_output: str) -> list:
    if not raw_output:
        return []
    try:
        text = _strip_think_tokens(raw_output.strip())
        return [s for s in json.loads(text).get("spans", [])
                if isinstance(s, str) and s.strip()]
    except (json.JSONDecodeError, AttributeError, TypeError):
        return []


def generate_pseudo_labels(
    sample_notes, train_df, features_df,
    faiss_index, faiss_metadata, model, tokenizer, sampling_params, cfg,
    draft_model=None,
) -> pd.DataFrame:
    feat_map      = features_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()
    case_features = features_df.groupby("case_num")["feature_num"].agg(list).to_dict()

    # ── Resume from checkpoint ────────────────────────────────────────────────
    done_ids, aug_rows = _load_checkpoint(cfg)
    rows_since_ckpt    = 0

    log.info("Loading sentence-transformer for FAISS retrieval ...")
    embed_model = SentenceTransformer(cfg["EMBED_MODEL"])

    # Count total pairs (for accurate progress bar)
    total_pairs = sum(
        len(case_features.get(int(r["case_num"]), []))
        for _, r in sample_notes.iterrows()
        if isinstance(r["pn_history"], str) and r["pn_history"].strip()
    )
    log.info(f"  Total pairs: {total_pairs} | already done: {len(done_ids)}")

    pbar = tqdm(total=total_pairs, initial=len(done_ids), desc="Generating", unit="pair")

    for _, note_row in sample_notes.iterrows():
        pn_num     = int(note_row["pn_num"])
        case_num   = int(note_row["case_num"])
        pn_history = note_row["pn_history"]
        if not isinstance(pn_history, str) or not pn_history.strip():
            continue

        for feature_num in case_features.get(case_num, []):
            feature_text = feat_map.get((case_num, feature_num), "")
            if not feature_text:
                continue

            row_id = f"{pn_num:05d}_{feature_num:03d}"
            if row_id in done_ids:
                pbar.update(1)
                continue  # skip already-done rows

            # Build + infer inline (no pre-accumulation = no RAM spike)
            try:
                few_shot = retrieve_few_shot_examples(
                    feature_text, faiss_index, faiss_metadata, embed_model, cfg["TOP_K_EXAMPLES"]
                )
                msgs   = build_messages(feature_text, pn_history, few_shot)
                output = _infer_one(msgs, model, tokenizer, sampling_params, draft_model=draft_model)
                spans  = _parse_spans(output)
            except Exception as exc:
                log.warning(f"  Skipped {row_id}: {exc}")
                spans = []

            locations      = find_span_locations(spans, pn_history, cfg["FUZZY_SCORE_CUTOFF"])
            annotation_col = spans     if spans     else [""]
            location_col   = locations if locations else [""]

            aug_rows.append({
                "id":          row_id,
                "pn_num":      pn_num,
                "feature_num": feature_num,
                "case_num":    case_num,
                "annotation":  str(annotation_col),
                "location":    str(location_col),
            })
            done_ids.add(row_id)
            rows_since_ckpt += 1
            pbar.update(1)

            if rows_since_ckpt >= cfg["CHECKPOINT_EVERY"]:
                _save_checkpoint(aug_rows, cfg)
                rows_since_ckpt = 0

    pbar.close()
    del embed_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    _save_checkpoint(aug_rows, cfg)   # final save

    aug_df    = pd.DataFrame(aug_rows)
    non_empty = (aug_df["location"] != str([""])).sum()
    fill_rate = 100.0 * non_empty / max(len(aug_df), 1)
    log.info(f"Done — {len(aug_df)} rows | non-empty: {non_empty} ({fill_rate:.1f}%)")
    return aug_df

print("✓ Section 6: generate_pseudo_labels defined (streaming + checkpoint)")


✓ Section 6: generate_pseudo_labels defined (streaming + checkpoint)


In [12]:
# Pre-cleanup before Phase 1
print("▶ Pre-cleanup: clearing GPU memory...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
print("✓ GPU memory cleared")


▶ Pre-cleanup: clearing GPU memory...
✓ GPU memory cleared


In [ ]:
def main():
    cfg       = CONFIG
    model       = None
    tokenizer   = None
    draft_model = None
    try:
        print("\n" + "="*65)
        print("  PHASE 1: Pseudo-Label Generation")
        print("="*65 + "\n")

        # Step 1: Load data
        print("▶ Step 1/4 — Loading and filtering data ...")
        sample_notes, train_df, features_df, pn_df = load_and_filter_data(cfg)
        print(f"  ✓ Notes: {len(sample_notes)} | Train: {len(train_df)} | Features: {len(features_df)}")

        # Step 2: FAISS index (cached to Drive)
        print("\n▶ Step 2/4 — Building / loading FAISS index ...")
        faiss_index, faiss_metadata = build_faiss_index(train_df, pn_df, features_df, cfg)
        print(f"  ✓ Index: {faiss_index.ntotal} vectors")

        # Step 3: Load model (4-bit, ~5 GB on T4)
        print("\n▶ Step 3/4 — Loading model (4-bit, ~5 GB VRAM) ...")
        model, tokenizer, draft_model = init_llm(cfg)
        sampling_params  = make_sampling_params(cfg)

        # Step 4: Generate — auto-resumes from checkpoint on Drive
        ckpt = Path(cfg["CHECKPOINT_FILE"])
        if ckpt.exists():
            done = len(pd.read_csv(ckpt))
            print(f"\n▶ Step 4/4 — Resuming from checkpoint ({done} rows already done) ...")
        else:
            print("\n▶ Step 4/4 — Generating pseudo-labels ...")
            print(f"  Checkpoint will save to: {cfg['CHECKPOINT_FILE']}")

        aug_df = generate_pseudo_labels(
            sample_notes, train_df, features_df,
            faiss_index, faiss_metadata, model, tokenizer, sampling_params, cfg,
            draft_model=draft_model,
        )

        # Save final CSV
        out_path = cfg["OUTPUT_FILE"]
        aug_df.to_csv(out_path, index=False)
        non_empty = (aug_df["location"] != str([""])).sum()
        fill_rate = 100.0 * non_empty / max(len(aug_df), 1)
        print(f"\n✓ Saved → {out_path}  shape={aug_df.shape}")
        print(f"  Non-empty labels: {non_empty} ({fill_rate:.1f}%)")

        # Delete checkpoint on clean completion
        if ckpt.exists():
            ckpt.unlink()
            print("  ✓ Checkpoint removed (run complete)")

        # Auto-download output CSV to local machine
        try:
            from google.colab import files
            print("\n▶ Downloading augmented_train.csv to your machine ...")
            files.download(str(out_path))
            print("  ✓ Download triggered")
        except Exception as e:
            print(f"  ⚠ Auto-download failed ({e}) — manually download from: {out_path}")

        print("\n" + "="*65)
        print("  ✓ Phase 1 complete — augmented_train.csv ready for Phase 2")
        print("="*65)

    except (KeyboardInterrupt, Exception) as e:
        print(f"\n⚠ Interrupted/error: {e}")
        print("  Progress saved to checkpoint — rerun main() to resume from where you left off.")
        if not isinstance(e, KeyboardInterrupt):
            raise
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        if draft_model is not None:
            del draft_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print("\n  ✓ Resources cleaned up")

main()



  PHASE 1: Pseudo-Label Generation

▶ Step 1/4 — Loading and filtering data ...
2026-04-26 19:38:44,458  [INFO]  Loading CSV files ...
2026-04-26 19:38:44,923  [INFO]    Annotated notes in train.csv : 1000
2026-04-26 19:38:44,952  [INFO]    Unannotated notes available  : 41146
2026-04-26 19:38:44,953  [INFO]    Sampled for labelling        : 500
  ✓ Notes: 500 | Train: 14300 | Features: 143

▶ Step 2/4 — Building / loading FAISS index ...
2026-04-26 19:38:44,954  [INFO]  Loading cached FAISS index ...
2026-04-26 19:38:45,006  [INFO]    Loaded 9901 vectors (dim=384)
  ✓ Index: 9901 vectors

▶ Step 3/4 — Loading model (4-bit, ~5 GB VRAM) ...
2026-04-26 19:38:45,006  [INFO]  Loading tokenizer: Qwen/Qwen3-8B ...
2026-04-26 19:38:45,529  [INFO]  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-26 19:38:45,600  [INFO]  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b968826d9c46dd6066d

model.safetensors.index.json: 0.00B [00:00, ?B/s]

2026-04-26 19:38:50,460  [INFO]  HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-8B/revision/main "HTTP/1.1 200 OK"


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

2026-04-26 19:38:50,794  [INFO]  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/b968826d9c46dd6066d109eabc6255188de91218/model-00001-of-00005.safetensors "HTTP/1.1 302 Found"
2026-04-26 19:38:50,921  [INFO]  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/b968826d9c46dd6066d109eabc6255188de91218/model-00004-of-00005.safetensors "HTTP/1.1 302 Found"
2026-04-26 19:38:50,938  [INFO]  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/b968826d9c46dd6066d109eabc6255188de91218/model-00003-of-00005.safetensors "HTTP/1.1 302 Found"
2026-04-26 19:38:50,942  [INFO]  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/b968826d9c46dd6066d109eabc6255188de91218/model-00002-of-00005.safetensors "HTTP/1.1 302 Found"
2026-04-26 19:38:51,085  [INFO]  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/b968826d9c46dd6066d109eabc6255188de91218/model-00005-of-00005.safetensors "HTTP/1.1 302 Found"
2026-04-26 19:38:51,113  [INFO]  HTTP Re